## 1️⃣ GPU Kontrolü ve Kurulum

In [2]:
# 🔍 GPU Kontrolü
import torch

print("🔍 GPU KONTROLÜ")
print("=" * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU Bulundu: {gpu_name}")
    print(f"💾 GPU Bellek: {gpu_memory:.1f} GB")
    print(f"🔥 CUDA Version: {torch.version.cuda}")
else:
    print("❌ GPU bulunamadı!")
    print("💡 Runtime > Change runtime type > GPU seçin")

🔍 GPU KONTROLÜ
❌ GPU bulunamadı!
💡 Runtime > Change runtime type > GPU seçin


In [3]:
# 📦 Gerekli Kütüphaneleri Yükle
print("📦 KÜTÜPHANELER YÜKLENİYOR...")
print("=" * 50)

!pip install -q transformers accelerate bitsandbytes
!pip install -q httpx beautifulsoup4 duckduckgo-search wikipedia
!pip install -q sentencepiece protobuf

print("\n✅ Tüm kütüphaneler yüklendi!")

📦 KÜTÜPHANELER YÜKLENİYOR...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 24.6 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 82.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.4 MB/s eta 0:00:00

✅ Tüm kütüphaneler yüklendi!


In [4]:
# 📥 GitHub'dan Projeyi İndir
import os

print("📥 PROJE İNDİRİLİYOR...")
print("=" * 50)

# Repo'yu klonla
if not os.path.exists('yapay-zeka-sistemi'):
    !git clone https://github.com/cebrailbagatarhan/yapay-zeka-sistemi.git
    print("✅ Proje indirildi!")
else:
    print("✅ Proje zaten mevcut, güncelleniyor...")
    %cd yapay-zeka-sistemi
    !git pull
    %cd ..

# Proje dizinine geç
%cd yapay-zeka-sistemi
print(f"\n📂 Çalışma dizini: {os.getcwd()}")

📥 PROJE İNDİRİLİYOR...
Cloning into 'yapay-zeka-sistemi'...
fatal: could not read Username for 'https://github.com': No such device or address
✅ Proje indirildi!
[Errno 2] No such file or directory: 'yapay-zeka-sistemi'
/content

📂 Çalışma dizini: /content


## 2️⃣ Qwen Modelini İndir ve Yükle

In [5]:
# 🤖 Qwen Modelini İndir
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print("🤖 QWEN 2.5-1.5B MODEL YÜKLENİYOR...")
print("=" * 50)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit Quantization config (GPU bellek tasarrufu)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("📥 Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

print("📥 Model yükleniyor (4-bit quantized)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

# Bellek bilgisi
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"\n✅ Model yüklendi!")
    print(f"💾 GPU Bellek Kullanımı: {allocated:.2f} GB")
    print(f"🚀 4-bit quantization ile ~%75 bellek tasarrufu!")

🤖 QWEN 2.5-1.5B MODEL YÜKLENİYOR...
📥 Tokenizer yükleniyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

📥 Model yükleniyor (4-bit quantized)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
# 🧪 Model Testi
def ask_qwen(prompt, max_tokens=256):
    """Qwen'e soru sor"""
    messages = [
        {"role": "system", "content": "Sen yardımcı bir asistansın. Türkçe ve İngilizce konuşabilirsin."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1
        )
    
    response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    return response

# Test
print("🧪 MODEL TESTİ")
print("=" * 50)

test_prompts = [
    "Merhaba! Nasılsın?",
    "Python'da liste nasıl oluşturulur?",
    "5 + 3 * 2 kaç eder?"
]

for prompt in test_prompts:
    print(f"\n👤 Soru: {prompt}")
    response = ask_qwen(prompt)
    print(f"🤖 Qwen: {response}")
    print("-" * 50)

🧪 MODEL TESTİ

👤 Soru: Merhaba! Nasılsın?


KeyboardInterrupt: 

## 3️⃣ Derin Web Araştırma Sistemi

In [7]:
# 🔍 Derin Web Araştırma Sistemi
import sys
sys.path.append('/content/yapay-zeka-sistemi')

from src.deep_web_researcher import DeepWebResearcher

print("🔍 DERİN WEB ARAŞTIRMA SİSTEMİ")
print("=" * 50)

# Araştırmacıyı başlat
researcher = DeepWebResearcher()
print("✅ Web araştırma sistemi hazır!")

# Test araştırması
def deep_research_with_qwen(topic, max_sources=5):
    """Derin araştırma + Qwen analizi"""
    print(f"\n🔍 ARAŞTIRMA: {topic}")
    print("=" * 50)
    
    # Web araştırması
    print("📡 Web taraması yapılıyor (async)...")
    results = researcher.deep_research(topic, max_sources=max_sources, use_async=True)
    
    if not results:
        print("❌ Sonuç bulunamadı")
        return
    
    print(f"\n✅ {len(results)} kaynak bulundu!\n")
    
    # Kaynakları göster
    print("📚 KAYNAKLAR:")
    for i, r in enumerate(results[:3], 1):
        print(f"{i}. {r['title'][:60]}...")
        print(f"   🔗 {r['url'][:50]}...")
    
    # Qwen ile özet
    print("\n🤖 Qwen analiz ediyor...")
    
    combined_info = f"'{topic}' hakkında bilgiler:\n\n"
    for r in results[:5]:
        combined_info += f"- {r['title']}: {r.get('snippet', '')[:200]}\n"
    
    prompt = f"{combined_info}\n\nYukarıdaki bilgilere dayanarak '{topic}' hakkında kısa bir özet yaz:"
    
    summary = ask_qwen(prompt, max_tokens=300)
    
    print("\n" + "=" * 50)
    print("📊 QWEN ÖZETİ:")
    print("=" * 50)
    print(summary)
    
    return summary

ModuleNotFoundError: No module named 'src'

In [8]:
# 🔬 Araştırma Testi
deep_research_with_qwen("Machine Learning nedir", max_sources=5)

NameError: name 'deep_research_with_qwen' is not defined

## 4️⃣ İnteraktif Sohbet Modu

In [ ]:
# 💬 İnteraktif Sohbet
from IPython.display import clear_output

print("💬 İNTERAKTİF SOHBET MODU")
print("=" * 50)
print("\n📋 Komutlar:")
print("  • Direkt soru yazın - Qwen cevaplar")
print("  • 'araştır: [konu]' - Web araştırması yapar")
print("  • 'q' - Çıkış")
print("\n" + "=" * 50)

conversation_history = []

while True:
    user_input = input("\n👤 Siz: ").strip()
    
    if user_input.lower() in ['q', 'quit', 'exit', 'çıkış']:
        print("\n👋 Görüşmek üzere!")
        break
    
    if not user_input:
        continue
    
    if user_input.lower().startswith("araştır:"):
        topic = user_input[8:].strip()
        if topic:
            deep_research_with_qwen(topic, max_sources=5)
    else:
        # Direkt soru
        response = ask_qwen(user_input)
        print(f"\n🤖 Qwen: {response}")
        
        # Geçmişe ekle
        conversation_history.append({"user": user_input, "assistant": response})

💬 İNTERAKTİF SOHBET MODU

📋 Komutlar:
  • Direkt soru yazın - Qwen cevaplar
  • 'araştır: [konu]' - Web araştırması yapar
  • 'q' - Çıkış



## 5️⃣ Bonus: Gradio Arayüzü

In [ ]:
# 🎨 Gradio Web Arayüzü
!pip install -q gradio

import gradio as gr

def chat_interface(message, history):
    """Gradio chat interface"""
    if message.lower().startswith("araştır:"):
        topic = message[8:].strip()
        results = researcher.deep_research(topic, max_sources=3, use_async=True)
        
        if results:
            info = f"'{topic}' hakkında {len(results)} kaynak bulundu:\n\n"
            for r in results[:3]:
                info += f"• {r['title']}: {r.get('snippet', '')[:100]}...\n"
            
            prompt = f"{info}\n\nBu bilgilere dayanarak özet yap:"
            response = ask_qwen(prompt, max_tokens=300)
        else:
            response = "Araştırma sonucu bulunamadı."
    else:
        response = ask_qwen(message)
    
    return response

# Gradio arayüzü
demo = gr.ChatInterface(
    fn=chat_interface,
    title="🤖 Yapay Zeka Sistemi",
    description="Qwen 2.5-1.5B + Derin Web Araştırma\n\n💡 'araştır: [konu]' yazarak web araştırması yapabilirsiniz.",
    examples=[
        "Merhaba! Nasılsın?",
        "Python'da döngü nasıl yazılır?",
        "araştır: Yapay zeka nedir"
    ],
    theme="soft"
)

print("🎨 Gradio arayüzü başlatılıyor...")
demo.launch(share=True, debug=False)

## 6️⃣ Model Eğitimi (Fine-Tuning)

Aşağıdaki hücrelerde Qwen modelini özel veri setlerimizle eğiteceğiz:
- **Conversational Dataset**: Sohbet ve yardım konuşmaları
- **Reasoning Dataset**: Teknik açıklamalar ve Chain-of-Thought
- **CoT Dataset**: Matematik ve mantık problemleri

In [ ]:
# 📦 Fine-Tuning için Gerekli Kütüphaneleri Yükle
print("📦 FINE-TUNING KÜTÜPHANELERİ YÜKLENİYOR...")
print("=" * 50)

!pip install -q peft trl datasets accelerate
!pip install -q bitsandbytes>=0.41.0
!pip install -q scipy

print("\n✅ Fine-tuning kütüphaneleri yüklendi!")
print("   • PEFT (LoRA için)")
print("   • TRL (Trainer)")
print("   • Datasets (Veri yönetimi)")
print("   • Accelerate (GPU optimizasyonu)")

In [ ]:
# 📚 Veri Setlerini Yükle ve Hazırla
import json
import os
from datasets import Dataset

print("📚 VERİ SETLERİ HAZIRLANIYOR...")
print("=" * 50)

# Veri setlerini yükle
def load_json_data(filepath):
    """JSON dosyasını yükle"""
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []

# Proje dizininde mi kontrol et
base_path = '/content/yapay-zeka-sistemi' if os.path.exists('/content/yapay-zeka-sistemi') else '.'

# Veri setlerini yükle
conversational_data = load_json_data(f'{base_path}/data/training/conversational_dataset.json')
reasoning_data = load_json_data(f'{base_path}/data/training/reasoning_chat_dataset.json')
cot_data = load_json_data(f'{base_path}/data/examples/cot_dataset.json')

print(f"✅ Conversational Dataset: {len(conversational_data)} örnek")
print(f"✅ Reasoning Dataset: {len(reasoning_data)} örnek")
print(f"✅ CoT Dataset: {len(cot_data)} örnek")

# Tüm verileri birleştir ve formatla
all_training_data = []

# Conversational data formatla
for item in conversational_data:
    all_training_data.append({
        'instruction': item.get('question', ''),
        'input': '',
        'output': item.get('response', ''),
        'type': 'conversational'
    })

# Reasoning data formatla
for item in reasoning_data:
    reasoning_text = item.get('reasoning', '')
    answer_text = item.get('answer', '')
    full_response = f"Düşünce Süreci:\n{reasoning_text}\n\nSonuç: {answer_text}"
    all_training_data.append({
        'instruction': item.get('question', ''),
        'input': '',
        'output': full_response,
        'type': 'reasoning'
    })

# CoT data formatla
for item in cot_data:
    reasoning_text = item.get('reasoning', '')
    answer_text = item.get('answer', '')
    full_response = f"Adım adım çözüm:\n{reasoning_text}\n\nCevap: {answer_text}"
    all_training_data.append({
        'instruction': item.get('question', ''),
        'input': '',
        'output': full_response,
        'type': 'cot'
    })

print(f"\n🎯 Toplam Eğitim Verisi: {len(all_training_data)} örnek")

# Dataset oluştur
train_dataset = Dataset.from_list(all_training_data)
print(f"✅ Dataset oluşturuldu: {train_dataset}")

# Örnek göster
print("\n📝 Örnek Veri:")
print("-" * 50)
sample = all_training_data[0]
print(f"Soru: {sample['instruction'][:100]}...")
print(f"Cevap: {sample['output'][:150]}...")

In [ ]:
# 🔧 LoRA Konfigürasyonu (Hafif Fine-Tuning)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("🔧 LoRA KONFİGÜRASYONU AYARLANIYOR...")
print("=" * 50)

# LoRA parametreleri
lora_config = LoraConfig(
    r=16,                          # LoRA rank (düşük = daha hafif)
    lora_alpha=32,                 # LoRA alpha (scaling factor)
    target_modules=[               # Hedef katmanlar
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj"       # MLP
    ],
    lora_dropout=0.05,             # Dropout
    bias="none",                   # Bias eğitilmez
    task_type="CAUSAL_LM"          # Dil modeli görevi
)

print("📊 LoRA Parametreleri:")
print(f"   • Rank (r): 16")
print(f"   • Alpha: 32")
print(f"   • Dropout: 0.05")
print(f"   • Target Modules: Attention + MLP katmanları")

# Modeli LoRA için hazırla
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Eğitilebilir parametre sayısını göster
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
percentage = 100 * trainable_params / total_params

print(f"\n✅ LoRA Model Hazır!")
print(f"📊 Eğitilebilir Parametreler: {trainable_params:,}")
print(f"📊 Toplam Parametreler: {total_params:,}")
print(f"📊 Eğitilen Yüzde: {percentage:.2f}%")
print(f"🚀 Bellek tasarrufu: ~{100-percentage:.1f}%!")

In [ ]:
# 📝 Veriyi Chat Formatına Dönüştür
def format_instruction(sample):
    """Qwen chat formatına dönüştür"""
    messages = [
        {"role": "system", "content": "Sen yardımcı bir Türkçe yapay zeka asistanısın. Adım adım düşünerek açık ve detaylı cevaplar verirsin."},
        {"role": "user", "content": sample['instruction']},
        {"role": "assistant", "content": sample['output']}
    ]
    
    # Chat template uygula
    formatted = tokenizer.apply_chat_template(
        messages, 
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted}

print("📝 VERİ FORMATLANIYOR...")
print("=" * 50)

# Dataset'i formatla
formatted_dataset = train_dataset.map(format_instruction)

print(f"✅ {len(formatted_dataset)} örnek formatlandı!")

# Örnek göster
print("\n📄 Formatlanmış Örnek:")
print("-" * 50)
print(formatted_dataset[0]['text'][:500] + "...")

In [ ]:
# 🚀 Eğitim Ayarları ve Trainer
from transformers import TrainingArguments
from trl import SFTTrainer

print("🚀 EĞİTİM AYARLARI HAZIRLANIYOR...")
print("=" * 50)

# Eğitim parametreleri
training_args = TrainingArguments(
    output_dir="./qwen-turkish-finetuned",  # Çıktı dizini
    num_train_epochs=3,                      # Epoch sayısı
    per_device_train_batch_size=2,           # Batch size (GPU'ya göre ayarla)
    gradient_accumulation_steps=4,           # Gradient biriktirme
    learning_rate=2e-4,                      # Öğrenme oranı
    weight_decay=0.01,                       # Weight decay
    warmup_ratio=0.1,                        # Warmup
    lr_scheduler_type="cosine",              # Scheduler
    logging_steps=10,                        # Log adımı
    save_steps=100,                          # Kaydetme adımı
    save_total_limit=2,                      # Max checkpoint
    fp16=True,                               # Mixed precision
    optim="paged_adamw_8bit",               # 8-bit optimizer
    gradient_checkpointing=True,             # Bellek tasarrufu
    report_to="none",                        # Logging
    max_grad_norm=0.3,                       # Gradient clipping
)

# SFT Trainer oluştur
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
)

print("✅ Trainer hazır!")
print("\n📊 Eğitim Parametreleri:")
print(f"   • Epochs: 3")
print(f"   • Batch Size: 2 (effective: 8)")
print(f"   • Learning Rate: 2e-4")
print(f"   • Max Sequence Length: 512")
print(f"   • Optimizer: Paged AdamW 8-bit")
print(f"   • FP16: Aktif")
print(f"   • Gradient Checkpointing: Aktif")

In [ ]:
# 🎯 EĞİTİMİ BAŞLAT!
import time

print("🎯 MODEL EĞİTİMİ BAŞLIYOR!")
print("=" * 50)
print("⏳ Bu işlem birkaç dakika sürebilir...")
print("📊 Eğitim ilerlemesini aşağıda takip edin:\n")

# GPU bellek durumu
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"💾 Başlangıç GPU Bellek: {allocated:.2f} GB\n")

start_time = time.time()

# Eğitimi başlat
trainer.train()

end_time = time.time()
training_time = end_time - start_time

print("\n" + "=" * 50)
print("✅ EĞİTİM TAMAMLANDI!")
print(f"⏱️ Toplam Süre: {training_time/60:.1f} dakika")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"💾 Final GPU Bellek: {allocated:.2f} GB")

In [ ]:
# 💾 Eğitilmiş Modeli Kaydet
print("💾 MODEL KAYDEDİLİYOR...")
print("=" * 50)

# LoRA adaptörlerini kaydet
model.save_pretrained("./qwen-turkish-finetuned")
tokenizer.save_pretrained("./qwen-turkish-finetuned")

print("✅ Model kaydedildi: ./qwen-turkish-finetuned")

# Google Drive'a kaydet (opsiyonel)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    import shutil
    drive_path = '/content/drive/MyDrive/AI-Models/qwen-turkish-finetuned'
    
    if not os.path.exists('/content/drive/MyDrive/AI-Models'):
        os.makedirs('/content/drive/MyDrive/AI-Models')
    
    if os.path.exists(drive_path):
        shutil.rmtree(drive_path)
    
    shutil.copytree("./qwen-turkish-finetuned", drive_path)
    print(f"✅ Google Drive'a kaydedildi: {drive_path}")
except:
    print("ℹ️ Google Drive bağlantısı yapılmadı (opsiyonel)")

In [ ]:
# 🧪 Eğitilmiş Modeli Test Et
print("🧪 EĞİTİLMİŞ MODEL TESTİ")
print("=" * 50)

def ask_finetuned_model(prompt, max_tokens=256):
    """Fine-tuned modele soru sor"""
    messages = [
        {"role": "system", "content": "Sen yardımcı bir Türkçe yapay zeka asistanısın. Adım adım düşünerek açık ve detaylı cevaplar verirsin."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1
        )
    
    response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    return response

# Test soruları
test_questions = [
    "Merhaba! Bana Python'da döngüler hakkında bilgi verir misin?",
    "5 + 3 × 2 kaç eder? Adım adım açıkla.",
    "Machine learning nedir? Basitçe açıklar mısın?",
    "API nedir ve neden kullanılır?"
]

print("\n📊 EĞİTİM ÖNCESİ vs SONRASI KARŞILAŞTIRMASI\n")

for i, question in enumerate(test_questions, 1):
    print(f"{'='*60}")
    print(f"📌 SORU {i}: {question}")
    print(f"{'='*60}")
    
    response = ask_finetuned_model(question)
    print(f"\n🤖 Fine-tuned Model Cevabı:")
    print(f"{response}")
    print()

## 7️⃣ Eğitim Metrikleri ve Analiz

In [ ]:
# 📊 Eğitim Grafiklerini Çiz
import matplotlib.pyplot as plt

print("📊 EĞİTİM METRİKLERİ")
print("=" * 50)

# Eğitim loglarını al
training_logs = trainer.state.log_history

# Loss değerlerini çıkar
train_losses = []
steps = []

for log in training_logs:
    if 'loss' in log:
        train_losses.append(log['loss'])
        steps.append(log.get('step', len(steps)))

# Grafik çiz
if train_losses:
    plt.figure(figsize=(12, 4))
    
    # Loss grafiği
    plt.subplot(1, 2, 1)
    plt.plot(steps, train_losses, 'b-', linewidth=2, label='Training Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('🎯 Training Loss Over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Loss dağılımı
    plt.subplot(1, 2, 2)
    plt.hist(train_losses, bins=20, color='steelblue', edgecolor='white')
    plt.xlabel('Loss Value')
    plt.ylabel('Frequency')
    plt.title('📈 Loss Distribution')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_metrics.png', dpi=150)
    plt.show()
    
    # İstatistikler
    print(f"\n📊 Eğitim İstatistikleri:")
    print(f"   • Başlangıç Loss: {train_losses[0]:.4f}")
    print(f"   • Final Loss: {train_losses[-1]:.4f}")
    print(f"   • İyileşme: {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.1f}%")
    print(f"   • Min Loss: {min(train_losses):.4f}")
    print(f"   • Ortalama Loss: {sum(train_losses)/len(train_losses):.4f}")
else:
    print("⚠️ Eğitim logları bulunamadı")

In [ ]:
# 🎮 İnteraktif Fine-tuned Model Sohbeti
print("🎮 FINE-TUNED MODEL İLE SOHBET")
print("=" * 50)
print("\n💡 Artık eğitilmiş modelinizle sohbet edebilirsiniz!")
print("📋 Komutlar:")
print("   • Direkt soru yazın")
print("   • 'q' ile çıkış")
print("\n" + "=" * 50)

while True:
    user_input = input("\n👤 Siz: ").strip()
    
    if user_input.lower() in ['q', 'quit', 'exit', 'çıkış']:
        print("\n👋 İyi çalışmalar!")
        break
    
    if not user_input:
        continue
    
    response = ask_finetuned_model(user_input)
    print(f"\n🤖 Model: {response}")

---

## 📊 Performans Bilgileri

| Özellik | Değer |
|---------|-------|
| Model | Qwen2.5-1.5B-Instruct |
| Quantization | 4-bit NF4 |
| GPU Bellek | ~1.5 GB |
| Web Scraping | Async (httpx) |
| Hız | ~180 token/s |

---

## 💡 İpuçları

1. **GPU seçin**: Runtime > Change runtime type > T4 GPU
2. **Session süresiz değil**: Colab ücretsiz 12 saat limit
3. **Model kaydet**: Drive'a mount edip modeli kaydedin
4. **Share link**: Gradio arayüzü ile başkalarıyla paylaşın

---

🎉 **İyi çalışmalar!**